# 3.5 Exercise: what does an image cost, and what does it buy?

Notebook 3.1 put a mixture of experts beside a dense head that uses as many parameters per
image, on four kinds of photo. Notebook 3.2 drew what parameters and data buy as curves.
Here you measure both yourself, on a larger mix of seven kinds of image, and you measure
one thing more for every model: what one image costs it.

At the end you can:

- read a **Pareto front** of accuracy against the cost of one image from runs you
  measured, name the runs on it, and say why "the best trial" is half a question;
- read a curve over data share and width: does a wider head pull ahead as the data grows,
  and what does a smaller head give up;
- put scaling into a search space: share and width as axes, the cost logged on every
  trial, and a control of equal compute beside an architecture of your own.

## What is asked of you

1. **Part 1: run this notebook as it is.** A baseline, a curve over data and width, a
   mixture of experts with its control of equal compute, and a short search; every run
   drawn on one front. You change nothing; you read what comes out and answer the
   questions for yourself. Its search measures what one trial costs on your machine,
   which `make study LESSON=3` computes your budget from.
2. **Part 2: one study of your own**, in the folder [`exercises/`](exercises/): one idea,
   stated before you run, a design that can test it, a search, and a control of equal
   compute beside it. The class spreads over the open questions at the end of this
   notebook, and builds on each other's ideas.
3. **Hand in your runs** before the next lesson, with one command: `make share LESSON=3`.

**This is homework with long runs.** Every experiment below says what it cost on a recent
laptop (Apple M-series, CPU); Part 1 took 16 to 18 minutes there, in two runs. Start a cell, read
the next section while it runs, or go to lunch. Part 2 is a study of 30 to 60 minutes on
your machine. On a slow machine, set `SHARES` to `[0.1, 0.3]` and lower `N_TRIALS`: an
epoch is the same steps at every share, but a narrow head on fewer rows settles in far
fewer epochs (width 16: 17 on 10% of the rows, 111 on all of them).

Every run goes to `artefacts/optuna/studies.db`, as in lessons 1 and 2: Part 1 to
`lesson3-notebook` (the search) and `lesson3-notebook-runs` (every other run), Part 2 to
`lesson3-study-<your study>`. Every run of E1 to E3 starts from the seed `SEED` (0).
Each point is one run: a run takes minutes, so it is run once. Another seed moves a point
by a few tenths of a point of accuracy, and up to a point on a small share (three seeds
of three heads, measured once: 0.4, 1.0 and 0.4 points from the lowest to the highest);
a difference of that size between two runs is one another seed could turn around.

## The data: seven kinds of image, two vectors each

The mix holds every image of seven datasets: textures (DTD), satellite tiles (EuroSAT),
flowers, dishes (Food-101), traffic signs (GTSRB), galaxies and blood cells, 321 classes
in all. Every image was run once through the pretrained encoder `dinov2-small` (lesson
1), and the cache keeps two of its vectors, 384 numbers each:

- the **CLS** vector, the encoder's one summary of the image, the one `ImageVectorizer`
  gave you in lesson 1;
- the **patch mean**: the encoder also gives one vector per patch of 14 x 14 pixels, and
  the mean of those 256 vectors is a second summary of the image.

A model here reads one or both, and gives a score per class. The domain of an image is
in the data, but no model is given it.

A run trains on a **share** of the train rows: the same share of every class, so a run
on 10% sees every class, each with a tenth of its images. A share of 10% is a subset of
the rows of 30%. A **validation split**, 10% of every class, is cut from train once and
is the same for every run: every score in this notebook is on it. Test is left alone, for
the end of the course.

## What one image costs

Lesson 1 counted the parameters of a head. A parameter stored is not a parameter used:
the mixture of experts of 3.1 stores eight experts and runs one per image. Two
measures say what one image costs, and every run in this lesson logs both beside
`n_params`:

- **FLOPs per image**: the floating-point operations of one forward pass, per image. A
  `Linear(384, 64)` does a multiply and an add per weight, 2 x 384 x 64 = 49,152 FLOPs.
  A mixture of experts counts its router and the one expert that ran, not the experts it
  stores.
- **latency per image**: the measured time of one forward pass over a batch of 256
  images, divided by 256, in microseconds, on one CPU thread. FLOPs count arithmetic;
  latency is what the clock says, and the two need not agree (3.1.6).

In this notebook **compute** means FLOPs per image: a control of equal compute does as
many FLOPs per image. 3.2's training compute is that times every image a run trains on;
two runs that train on as many images get here the same steps.

## The Pareto front

A model is better than another when it scores higher and costs less. Most pairs are
neither: one scores higher and costs more. The **Pareto front** of a set of runs is the
runs that no other run beats on both: no run is at least as accurate and cheaper. Every
run off the front has a run on it that is at least as good on both measures and better
on one. With FLOPs on the x-axis, better is up and to the left, and the cheapest run is
always on the front. "The best trial" picks the highest score and ignores the cost; the
front shows what each step in accuracy costs.

## Setup

The data, the trainer and the helpers (`run`, `table`, `plot_front`, `search`) come from
[`exercises/harness.py`](exercises/harness.py), the design `Mix` from
[`exercises/designs.py`](exercises/designs.py): the same code your study uses in Part 2.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from exercises.designs import DESIGN, Config, control_width
from exercises.harness import (
    DOMAINS,
    EPOCHS,
    HARNESS,
    HUES,
    LINEAR,
    NOTEBOOK,
    Linear,
    cut,
    load,
    plot_front,
    run,
    search,
    study_space,
    table,
)
from notebooktester import param

from hpo.measure import Architecture
from hpo.record import StudyLog
from hpo.search import Budget

WIDTHS = [16, 64, 256]  # the hidden widths of E2
SHARES = param([0.1, 0.3, 1.0], test=[0.1])  # the data shares of E2
N_TRIALS = param(24, test=2)  # the trials of E4
COLOURS = dict(zip(["E1", "E2", "E3", "E4"], HUES, strict=False))
# every choice of the design as Config gives it, and the trainer's: every train row of
# dinov2-small, as flat vectors
TRAINER = {
    "dataset": "mix7",
    "encoder": "dinov2-small",
    "share": 1.0,
    "drop_domain": "none",
    "form": "flat",
    "lr": 1e-3,
    "epochs": EPOCHS,
}
PLAIN = {**vars(Config()), **TRAINER}


def build(config: dict, data):
    """The model of the design that `config` chooses, at the sizes of `data`."""
    return DESIGN.build(
        config,
        dim=data.dims["dim"],
        height=data.dims["height"],
        breadth=data.dims["breadth"],
        n_classes=len(data.classes),
    )


mix = load("dinov2-small")
train_rows, valid_rows = cut("dinov2-small", 1.0)
overview = pd.DataFrame(
    {
        "classes": [
            len(np.unique(mix.label[mix.domain == d])) for d in range(len(DOMAINS))
        ],
        "train": np.bincount(mix.domain[train_rows], minlength=len(DOMAINS)),
        "validation": np.bincount(mix.domain[valid_rows], minlength=len(DOMAINS)),
    },
    index=DOMAINS,
)
overview.loc["all"] = overview.sum()
display(overview)
print(f"every image: {mix.embed.shape[1:]}, the CLS vector and the patch mean")

Food-101 holds almost half the train rows, DTD and the flowers few: a model good at food
and traffic signs scores high on accuracy, whatever it does on textures.

# Part 1: the notebook as handed out

How a run trains, in every experiment: Adam at a learning rate of 1e-3, batches of 256.
An **epoch** here is a fixed number of steps, one pass over all 148,090 train rows,
whatever the share: on a share of 10% an epoch passes over its rows ten times. So the
epochs of two runs are the same steps. A run stops once its validation accuracy has not
improved for `PATIENCE` (10) epochs in a row, or after `EPOCHS` (150), and keeps the
weights of its best epoch, the **kept epoch**.

## E1: the baseline, first

A linear layer on the CLS vector, every train row: `Linear(384, 321)`. It is the floor
every later number is read against, here and in your own study.

**Cost:** 1 minute, plus the first download of the mix.

In [ ]:
e1 = run("linear", {**LINEAR, **TRAINER}, "baseline", "E1",
         make=lambda config, data: Linear(data.dims["dim"], len(data.classes)))  # fmt: skip
table([e1])

A linear layer costs two FLOPs per weight: 2 x 384 x 321 = 246,528 per image. Keep that
number in mind: a head with a hidden layer can cost less than the linear layer, because
the hidden layer is narrow.

## E2: a curve over data and width

**Question.** Does a wider head pull ahead as the data grows? A head with one hidden
layer of width 16, 64 or 256, on 10%, 30% and all of the train rows of every class.

The head is the design of [`exercises/designs.py`](exercises/designs.py) with every
option off: the CLS vector, one hidden layer, the classifier. Open it in atlas before you
run it:

```bash
atlas-code-nightly view notebooks/3_scaling_and_search/exercises
```

<svg width="660" height="80" viewBox="0 0 660 80" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="12">
<defs><marker id="e2a" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 z" fill="#444"/></marker></defs>
<text x="4" y="44" fill="#222">image</text>
<line x1="46" y1="40" x2="128" y2="40" stroke="#444" stroke-width="1.5" marker-end="url(#e2a)"/>
<text x="88" y="32" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">(B, 2, 384)</text>
<rect x="130" y="22" width="90" height="36" rx="6" fill="#eeeeee" stroke="#555"/>
<text x="175" y="44" text-anchor="middle" fill="#222">choose CLS</text>
<line x1="220" y1="40" x2="290" y2="40" stroke="#444" stroke-width="1.5" marker-end="url(#e2a)"/>
<text x="255" y="32" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">(B, 384)</text>
<rect x="292" y="22" width="150" height="36" rx="6" fill="#cfe3f3" stroke="#555"/>
<text x="367" y="44" text-anchor="middle" fill="#222">Linear(384, w), GELU</text>
<line x1="442" y1="40" x2="508" y2="40" stroke="#444" stroke-width="1.5" marker-end="url(#e2a)"/>
<text x="475" y="32" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">(B, w)</text>
<rect x="510" y="22" width="110" height="36" rx="6" fill="#cfe3f3" stroke="#555"/>
<text x="565" y="44" text-anchor="middle" fill="#222">Linear(w, 321)</text>
<line x1="620" y1="40" x2="656" y2="40" stroke="#444" stroke-width="1.5" marker-end="url(#e2a)"/>
</svg>

- `(B, 2, 384)`: a batch of `B` images, each its CLS vector and its patch mean;
- `(B, 384)`: the CLS vector, row 0;
- `(B, w)`: the hidden layer, `w` numbers per image;
- `(B, 321)`: a score per class.

The cost of one image is 2 x (384 x w + w x 321) FLOPs: 22,560 at width 16, 360,960 at
width 256. Width 16 costs a tenth of the linear layer; width 256 costs more than it.

**Setup.** Nine runs, one per width and share, each trained until it settles, from the
same seed. Every share gets the same steps per epoch, so a run on 10% is not short of
steps, only of distinct images.

**Expect.** Before you run the next cell, write down: at 10% of the rows, which width
scores highest? At all of them? Does the gap between width 16 and width 256 grow or
shrink as the share grows?

**Cost:** 3 minutes, most of it the runs on every train row.

In [ ]:
e2 = [
    run(
        f"w{width} s{share:g}",
        {**PLAIN, "width": width, "share": share},
        "hypothesis",
        "E2",
        build,
    )
    for share in SHARES
    for width in WIDTHS
]
table(e2)

In [ ]:
fig, (curve, settle) = plt.subplots(1, 2, figsize=(13, 4))
hues = dict(zip(WIDTHS, ["#56B4E9", "#0072B2", "#CC79A7"], strict=True))
for width in WIDTHS:
    mine = [r for r in e2 if r["width"] == width]
    curve.plot([r["share"] for r in mine], [r["accuracy"] for r in mine], "o-",
               color=hues[width], label=f"width {width}")  # fmt: skip
    for r in mine:
        epochs = np.arange(1, len(r["curve"]) + 1)
        style = "-" if r["share"] == 1.0 else ":" if r["share"] == 0.1 else "--"
        settle.plot(epochs, r["curve"], style, color=hues[width], lw=1)
        settle.scatter(
            [r["kept_epoch"]],
            [r["curve"][r["kept_epoch"] - 1]],
            color=hues[width],
            s=18,
        )
curve.axhline(
    e1["accuracy"], color="black", ls="--", lw=1, label="linear, every row (E1)"
)
curve.set_xscale("log")
curve.set_xticks(SHARES, [f"{s:.0%}" for s in SHARES])
curve.minorticks_off()
curve.set_xlabel("share of the train rows of every class (log scale)")
curve.set_ylabel("validation accuracy")
curve.set_title(
    "the curve over data and width: one run per point", loc="left", fontsize=10
)
curve.legend(fontsize=8)
settle.set_xlabel("epoch (one epoch = the same steps at every share)")
settle.set_ylabel("validation accuracy")
settle.set_title("every run per epoch; dot = kept epoch; dotted 10%, dashed 30%, solid 100%",
                 loc="left", fontsize=10)  # fmt: skip
for ax in (curve, settle):
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

by = {(r["width"], r["share"]): r["accuracy"] for r in e2}
for share in SHARES:
    gap = 100 * (by[WIDTHS[-1], share] - by[WIDTHS[0], share])
    print(
        f"share {share:>4.0%}: width {WIDTHS[-1]} - width {WIDTHS[0]} = {gap:+.1f} points"
    )

Read the two panels:

1. **Has every run settled?** In the right panel, read each line before its dot: still
   climbing, flat, or turned down. A validation curve that turns down is the sign of
   overfitting from lesson 1: which width does it, at which shares?
2. **Does the wider head pull ahead as the data grows?** The printed gaps are width 256
   minus width 16 at each share. If they shrink, this is the bottleneck of 3.2.6, not the
   curve of 3.2.5: on frozen vectors a width limits how much of the vector reaches the
   classifier, the head does not learn the images, and the wide head is already close to
   the linear layer, so more data lifts the narrow head more.
3. **Against the linear layer.** The dashed line is E1. The widths approach it from
   below, as in 3.2.6, and at 100% the widest lies under it by less than a point. DINOv2's
   vectors are laid out for one linear layer (it is evaluated with linear probes), a
   hidden layer narrower than the 384 numbers of the vector can only pass on less, and
   these heads have no regularisation. A rule of thumb, which E2 does not test: an
   unregularised bigger head overfits; regularised, it ties with the smaller one, and
   what it then costs is FLOPs.
4. **What does a smaller head give up?** Read the table: width 16 against width 64 and
   256 at every share, in accuracy and in FLOPs. How many points does a tenth of the
   FLOPs cost?
5. **Where would you put the ranges of a search?** Where the curve still changes. Every
   width still rises from 30% to 100%, so the share axis runs to every row. Over width,
   16 to 64 buys 7 to 11 points and 64 to 256 buys 2 to 3: the bend lies between 64 and
   256, so a log axis from 16 to 256 puts trials on both sides of it, where one from 128
   up would spend them on the flat part.

## E3: a mixture of experts and its control of equal compute

**Question.** Notebook 3.1 found that, at the same parameters used per image, a sparse
mixture of experts beat a dense head on four domains. Does it on seven?

**The two runs.** The mixture of experts is 3.1's: 8 experts, each a head with one hidden
layer of width 8, and a router, `Linear(384, 8)`, that runs one of them per image (the
option `moe` of the design). Its **control of equal compute** is the plain head that does
as many FLOPs per image: the cell below counts the mixture's FLOPs before anything is
trained, and `control_width` gives the width of the plain head that matches. (3.1
matched the parameters used per image, which gives the same width here.)

**Expect.** Write down: will the mixture lie above its control on the front, at the same
FLOPs? By how much, against the gap between width 16 and width 64 in E2?

**Cost:** 4 minutes; the router's gather and scatter make every step of the
mixture slower than its FLOPs say.

In [ ]:
EXPERT_WIDTH, N_EXPERTS = 8, 8
moe_config = {**PLAIN, "moe": True, "width": EXPERT_WIDTH, "n_experts": N_EXPERTS}
data = HARNESS.data(PLAIN)
moe_flops = Architecture.of(build(moe_config, data), data.example).flops_per_image
control = control_width(moe_flops, d_in=384, n_classes=321)
control_config = {**PLAIN, "width": control}
for name, config in [
    (f"MoE 8 x {EXPERT_WIDTH}", moe_config),
    (f"dense {control}", control_config),
]:
    arch = Architecture.of(build(config, data), data.example)
    print(
        f"{name:>10s}: {arch.flops_per_image:>7,} FLOPs per image, {arch.n_params:>8,} parameters stored"
    )

In [ ]:
e3 = [
    run(f"MoE 8 x {EXPERT_WIDTH}", moe_config, "hypothesis", "E3", build),
    run(f"dense {control}", control_config, "control", "E3", build),
]
display(table(e3))
part1 = [e1, *e2, *e3]
on = plot_front(part1, "every run of E1 to E3: accuracy against what one image costs")
print("on the front:", ", ".join(r["name"] for r in on))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
for r in part1:
    ax.scatter(r["flops"], r["latency_us"], color=COLOURS[r["experiment"]], s=30)
    if r["share"] == 1.0:
        ax.annotate(r["name"], (r["flops"], r["latency_us"]), xytext=(4, 2),
                    textcoords="offset points", fontsize=8)  # fmt: skip
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("FLOPs per image (log scale)")
ax.set_ylabel("latency per image, µs (log scale)")
ax.set_title("does the clock agree with the FLOPs? one thread, batches of 256; "
             "unlabelled: E2 on 10% and 30%", loc="left", fontsize=10)  # fmt: skip
ax.spines[["top", "right"]].set_visible(False)
plt.show()

Read the front and the clock:

1. **Which runs are on the front?** Name them, from cheap to dear. For every run off the
   front, which run on it beats it on both measures?
2. **Did the mixture beat its control?** The two runs of E3 sit at about the same FLOPs.
   How many points lie between them, against the few tenths another seed moves a run? If
   the mixture is above, its gain comes from what it stores beyond what it uses, the seven
   other experts, since its control does as many FLOPs.
3. **What does the clock say?** Read the mixture's latency against its control's:
   sending every image to its expert and putting the scores back is work the FLOPs do not
   count. The linear layer is the fastest run, with ten times the FLOPs of width 16: one
   matrix product against two and an activation, and at this size a call costs overhead
   more than arithmetic. Fewer FLOPs is not automatically faster (3.1.6); a few
   hundredths of a µs between equal heads is the clock's noise.

## E4: a starter search, on the same front

**Question.** A search spends its trials over a space; where on the front do its trials
land? The space is `SPACE` of [`exercises/study.py`](exercises/study.py), the starter
space:

| axis | values | what it turns |
|---|---|---|
| `width` | 16 to 256 | the hidden layers: parameters and FLOPs |
| `depth` | 1 or 2 | hidden layers |
| `share` | 10% to 100% | data |
| `input` | CLS, patch mean, both | what the head reads: both doubles the first layer |
| `dropout` | 0 to 0.3 | regularisation: while training, that share of the hidden numbers is set to zero at random, so the head cannot lean on any one |

The search samples at random and prunes nothing: a pruner stops a trial on its first
epochs, and the small heads, which learn slowest at first, are the ones the front asks
about. It is not seeded, so the class's trials pooled are one random search of a few
hundred trials. Its trials measure what one trial costs on your machine, which is what
Part 2's budget is computed from.

**Expect.** Write down: will the trial with the highest accuracy be on the front? At
which end of the front will most trials land?

**Cost:** 9 to 11 minutes for 24 trials.

In [ ]:
STARTER_SPACE = study_space()
e4 = search(STARTER_SPACE, N_TRIALS, build, "E4")
pd.DataFrame(e4).sort_values("accuracy", ascending=False).drop(
    columns="experiment"
).style.format(
    {
        "accuracy": "{:.3f}",
        "flops": "{:,.0f}",
        "latency_us": "{:.2f}",
        "share": "{:.2f}",
        "dropout": "{:.2f}",
    }
).hide(axis="index")

In [ ]:
on = plot_front(
    part1 + e4, "Part 1 and the starter search: accuracy against what one image costs"
)
print("on the front:", ", ".join(r["name"] for r in on))
best = max(e4, key=lambda r: r["accuracy"])
print(f"the best trial: {best['name']}, accuracy {best['accuracy']:.3f}, "
      f"{best['flops']:,} FLOPs per image")  # fmt: skip

axes = ["width", "depth", "share", "input", "dropout"]
fig, panels = plt.subplots(1, len(axes), figsize=(17, 3), sharey=True)
trials = pd.DataFrame(e4)
for ax, axis in zip(panels, axes, strict=True):
    ax.scatter(trials[axis], trials["accuracy"], color=COLOURS["E4"], alpha=0.8)
    ax.set_xlabel(axis)
    if axis in ("width", "share"):
        ax.set_xscale("log")
        ax.minorticks_off()
        ticks = [16, 32, 64, 128, 256] if axis == "width" else [0.1, 0.2, 0.5, 1.0]
        ax.set_xticks(ticks, [f"{t:g}" for t in ticks])
    if axis == "depth":
        ax.set_xticks([1, 2])
    ax.spines[["top", "right"]].set_visible(False)
panels[0].set_ylabel("validation accuracy")
plt.suptitle("every trial of the search, one panel per axis", x=0.1, ha="left")
plt.tight_layout()
plt.show()

1. **Is the best trial on the front?** Only if no cheaper run of Part 1 scores as much:
   compare it with the linear layer of E1. Along the front, what does each step buy in
   points, for how many more FLOPs?
2. **Which trials are off the front?** Which axes do they share, in the table?
3. **Per axis:** which panels show a pattern on 24 trials? A panel shows one axis while
   the others vary: if the depth-2 trials lie low, read their width and share in the
   table, since a narrow head or a small share pulls a trial down whatever its depth.
   Whether depth pays, with regularisation, is the "Deeper heads" question of Part 2. An
   axis whose panel shows no pattern showed no effect on these draws, which the other
   axes may hide.

# Part 2: your own study

The loop is the one of lessons 1 and 2, on lesson 3's design: the nine steps of the
[methodology page](https://raoulg.github.io/MADS-MachineLearning-course/your-study.html#lesson-3),
the same in every lesson. You work in [`exercises/`](exercises/):

| file | what it is | what you do with it |
|---|---|---|
| [`exercises/designs.py`](exercises/designs.py) | `Mix`, one typed method per stage, and `Head` inside it; the options are `kernel` and `moe` on `Mix`, `skip` and `gate` on `Head` | read it with atlas; add a stage when your idea needs another block |
| [`exercises/study.py`](exercises/study.py) | six settings at the top (`STUDY`, `IDEA`, `EXPECTS`, `OBJECTIVE`, `SPACE`, `MINUTES`) and the code that runs the study | fill in the settings, then `make study LESSON=3` |
| [`exercises/harness.py`](exercises/harness.py) | the mix, `share`, `drop_domain`, `encoder`, `form`, the trainer that stops when a run settles, the linear baseline, and this notebook's helpers | nothing: it is given |

**A control of equal compute.** Without the plain head that does as many FLOPs per
image beside it, a gain of your design may be one of more FLOPs. FLOPs, not parameters:
a kernel stores few weights and does many FLOPs (lesson 2).

1. Count your design's FLOPs before you train, as E3 did:
   `Architecture.of(model, data.example).flops_per_image`;
   `control_width(flops, d_in=384, n_classes=321)` is the matching plain head's width.
2. Search your option as `Choice([False, True])` beside `width`, over a range that holds
   that width.
3. Draw your study's front with the two apart, in a cell of this notebook:

   ```python
   from exercises.harness import rows_of

   plot_front(rows_of("lesson3-study-<your study>", by="<your option>"), "my study")
   ```

   Your option pays where its trials lie above the others at the same FLOPs.

## The open questions

Every question below is one bet a study can test; the class spreads over them, so that
the next lesson opens on a map of what scale and architecture bought, each run on one
front, not a leaderboard. A bet that did not pay is a result on that map.

- **Can a hidden layer beat the linear layer?** In E2 the linear layer on CLS lies less
  than a point above width 256 at every share. Does a head wider than the 384 numbers of
  the vector, with `dropout` or `norm`, pass it, and at how many FLOPs per image?
- **The curves.** E2 measured three widths and three shares. Where does the curve over
  width bend at every row, and where over share? Does a deeper head (`depth`) shift the
  curve, or only its cost? Read both against FLOPs, not only against width.
- **How small can you go?** What is the cheapest head within one point of the best run,
  and within five? Which axis buys the most accuracy per FLOP at the cheap end of the
  front: width, input or depth?
- **Experts against their control.** E3 measured one mixture. Does a mixture of more
  experts, or wider ones, stay above its control of equal compute? Does the router split
  the images by domain? Every image carries its domain, so the expert a router picks per
  domain can be counted, as 3.1.5 counted it per dataset.
- **A kernel on the vector.** `form="grid"` gives every vector folded into a grid,
  `(B, 2, 16, 24)`, and the option `kernel` reads that grid with a 2D kernel and a pool.
  Lesson 2 folded a beat into a grid because neighbouring steps of a beat belong
  together. Is that bet right for a DINOv2 vector, whose neighbouring numbers are
  features the encoder put side by side? Read it against a plain head of the same FLOPs.
- **CLS, patch mean, or both?** Does the patch mean help some domains more than others
  (textures, galaxies)? Every run logs the recall of every class, `<domain>/<class>`.
  Is `both`, which doubles the first layer, worth its FLOPs against a wider head on CLS?
- **Deeper heads.** With `depth` 2 to 4, do `skip` and `gate` let a deeper head train?
  Does it beat a wide head of one layer at the same FLOPs, and with what regularisation
  (`dropout`, `norm`) and data? A highway (lesson 2) is a stage you can add.
- **Small against base.** `encoder="dinov2-base"` gives vectors of 768 numbers from a
  larger encoder. Does a better encoder buy more than a wider head, at the FLOPs of the
  head? The encoder's own cost per image is not in these FLOPs: say what that leaves out.
- **Leave a domain out.** `drop_domain` takes one domain out of train and validation, so
  that run is scored on six domains and another on seven. Compare the six shared domains
  in both, from the recall per class: does the rest score higher without it, and which
  domain costs the others most?

A question of your own is welcome, as long as it is a bet you can state before you run.

## The steps

Run every command in a terminal, from the root of the repository.

1. **Design.** Read the design in atlas:

   ```bash
   atlas-code-nightly view notebooks/3_scaling_and_search/exercises
   ```

2. **Build.** Change the design if your question needs a block it does not have, then check
   that every combination builds:

   ```bash
   uv run notebooks/3_scaling_and_search/exercises/designs.py
   ```

3. **Baseline.** Nothing to do: `make study LESSON=3` measures the linear baseline before it
   searches.
4. **Budget and space.** `STUDY` is your first name and what it studies:
   `"sam-kernel-on-the-fold"`. `IDEA` says what you investigate in your own words, and its
   `builds_on` names the idea yours starts from: Part 1 (`Idea.handed_out(3).name`) or an
   idea of the class, by its name. `EXPECTS` is what you expect, and why, written before
   you run. `SPACE` searches the axes your question needs, with the control of equal
   compute among its trials, and fixes the rest; every option of the design is an axis,
   searched or fixed. `MINUTES` is your budget; the cell below shows what it buys.
5. **Search:** `make study LESSON=3`. It searches, reruns the winner and prints it against
   the baseline in points: one end of your front.
6. **Read** your front with `rows_of` and `plot_front`, as above; `make optuna-dashboard`
   shows each trial's `flops_per_image` and `latency_us` under "User Attributes".
7. **Explain:** `make page LESSON=3` builds your page of the lesson from your store; its
   step on the page says what to fill in.
8. **Share:** `make share LESSON=3`.
9. **Build on:** see the class in `make dashboard LESSON=3`. Your next idea builds on one
   of the class.

In [ ]:
MINUTES = 30  # the time you give your study on this machine
AXES = len(STARTER_SPACE.searched())  # the axes the starter space searches
PART1 = StudyLog.load(name=NOTEBOOK)
SECONDS_PER_TRIAL = PART1.seconds_per_trial(epochs=EPOCHS)


def trials_in(minutes: float) -> int:
    """The trials that fit in `minutes`, once the baseline and the winner's rerun are set
    aside."""
    return Budget.at(
        minutes=minutes,
        seconds_per_trial=SECONDS_PER_TRIAL,
        measured_on=PART1.name,
        epochs=EPOCHS,
        repeats=HARNESS.repeats,
    ).n_trials


minutes = np.arange(1, 91)
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(minutes, [trials_in(m) for m in minutes], color=COLOURS["E2"])
ax.scatter([MINUTES], [trials_in(MINUTES)], color=COLOURS["E3"], s=80, zorder=3)
ax.annotate(
    f"{MINUTES} minutes: {trials_in(MINUTES)} trials, "
    f"{trials_in(MINUTES) / AXES:.0f} per axis of the starter space",
    (MINUTES, trials_in(MINUTES)),
    xytext=(10, -14),
    textcoords="offset points",
)
ax.set_xlabel("minutes you give the study")
ax.set_ylabel("trials that fit")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title(
    f"a trial of Part 1 took {SECONDS_PER_TRIAL:.0f}s here (the median of its trials)",
    loc="left",
    fontsize=10,
)
plt.show()

A trial of your space can cost more or less than one of the starter space: a run on a
small share settles in fewer epochs, a mixture of experts or a kernel takes longer per
step (E3's mixture took several times as long as its control). The first trials of your
study tell you; `make optuna-dashboard` shows their `train_seconds`.

## Hand in your runs before the next lesson

```bash
make share LESSON=3
```

It hands in Part 1 (`lesson3-notebook`, `lesson3-notebook-runs`) and every study of
yours (`lesson3-study-*`), each run with the design script it ran. Name the ideas of
classmates you built on in `builds_on`; the class page shows who built on whom.